https://arxiv.org/pdf/1804.03209

In [1]:
import torchaudio
data = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./")

print(str(torchaudio.list_audio_backends()))

['soundfile']


In [ ]:
print("Nombre d'échantillon : ", data.__len__())

Le dataset propose des ensemble pré-établi pour l'entraînement, la validation et le test.

In [2]:
data_train = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="training")
data_validation = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="validation")
data_testing = torchaudio.datasets.SPEECHCOMMANDS(download=True, root="./", subset="testing")

In [ ]:
print("Nombre d'échantillon d'entrainement : ", data_train.__len__())
print("Nombre d'échantillon de validation : ", data_validation.__len__())
print("Nombre d'échantillon de test : ", data_testing.__len__())

In [3]:
import os
labels = os.listdir("./SpeechCommands/speech_commands_v0.02/")

labels.remove("README.md")
labels.remove("LICENSE")
labels.remove("testing_list.txt")
labels.remove("validation_list.txt")
labels.remove("_background_noise_")
labels.remove(".DS_Store")

print("Il y a  : ", len(labels), " labels")
print(labels)

Il y a  :  35  labels
['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']


In [ ]:
import matplotlib.pyplot as plt

#Affichage de la distribution des ensembles proposé par le dataset
def distribution_dataset(dataset):
    count = [0 for i in range(len(labels))]

    for i in range(dataset.__len__()):
        count[labels.index(dataset.__getitem__(i)[2])] += 1
    
    plt.figure(figsize=(20,5))
    plt.xticks(rotation=90)
    plt.title("Distribution")
    plt.bar(labels, count)

In [ ]:
distribution_dataset(data_train)

In [ ]:
distribution_dataset(data_testing)

In [ ]:
distribution_dataset(data_validation)

In [ ]:
#Permet de récuperer un échantillon de chaque label

count = [0 for i in range(len(labels))]
sample = [[] for i in range(len(labels))]

for i in range(data_validation.__len__()):
    item = data_validation.__getitem__(i)
    index = labels.index(item[2])
    if count[index] == 0:
        sample[index] = item[0]
        count[index] = 1
    if sum(count) == 35:
        break
    

In [ ]:
fig, axs = plt.subplots(7,5, figsize=(20,10))

for i in range(len(sample)):
    axs[i // 5, i%5].set_title(labels[i])
    axs[i // 5, i%5].plot(sample[i][0])

plt.tight_layout()
plt.show()

In [ ]:
import torchaudio.transforms as T
import torch

fig, axs = plt.subplots(7,5, figsize=(20,20))
transform = T.MelSpectrogram(n_fft=1024, sample_rate=16000, n_mels=64, hop_length=512)

for i in range(len(sample)):
    spectrogram = transform(sample[i][0])
    axs[i // 5, i%5].imshow(spectrogram.numpy(), origin="lower", aspect="auto", interpolation="nearest")
    axs[i // 5, i%5].set_title(labels[i])

plt.tight_layout()
plt.show()

In [ ]:
import torchaudio.transforms as T
import torch

fig, axs = plt.subplots(7,5, figsize=(20,20))
transform = T.MelSpectrogram(n_fft=1024, sample_rate=16000, n_mels=64, hop_length=512)

for i in range(len(sample)):
    spectrogram = transform(sample[i][0])
    log_mel = torch.log(spectrogram + 1e-10)
    axs[i // 5, i%5].imshow(log_mel.numpy(), origin="lower", aspect="auto", interpolation="nearest")
    axs[i // 5, i%5].set_title(labels[i])

plt.tight_layout()
plt.show()

In [ ]:
irregulier_longueur_count = 0
irregulier_samplerate_count = 0

for i in data:
    if i[1] != 16000: irregulier_samplerate_count += 1
    if len(i[0][0]) != 16000: irregulier_longueur_count += 1

In [ ]:
print(irregulier_samplerate_count)

Les fichiers ont tous le même samplerate

In [ ]:
print(irregulier_longueur_count)

La plus part des fichiers ne font pas la même taille

In [ ]:
irregulier_grand_count = 0
irregulier_petit_count = 0

for i in data:
    if len(i[0][0])  < 16000: irregulier_petit_count += 1
    if len(i[0][0])  > 16000: irregulier_grand_count += 1

In [ ]:
print(irregulier_grand_count)

In [ ]:
print(irregulier_petit_count)

Donc tout les segments sont plus petits que la longueur max. On va donc rajoute du silence a la fin des audio pour rendre les données uniforme.

In [14]:
from torch.utils.data import DataLoader

label_to_idx = {l: i for i, l in enumerate(labels)}

def pre_process_batch(batch):
    target_len = 16000
    new_batch = []
    
    for waveform, sample_rate, label_str, speaker_id, utterance_number in batch:

        #Ici on rajoute du bruit blanc en entree
        current_len = waveform.shape[1]
        if current_len < target_len:
            waveform = torch.nn.functional.pad(waveform, (0, target_len - current_len))

        label = label_to_idx.get(label_str, None)
        if label is None:
            continue

        new_batch.append((waveform, label))

    waveforms = torch.stack([item[0] for item in new_batch])
    labels_tensor = torch.tensor([item[1] for item in new_batch])
    
    return waveforms, labels_tensor
 
train_loader = DataLoader(
    data_train, 
    batch_size=32, 
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False,
    collate_fn=pre_process_batch 
)

val_loader = DataLoader(
    data_validation, 
    batch_size=32, 
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=False,
    collate_fn=pre_process_batch 
)

#for batch_idx, (waveforms, labels_batch) in enumerate(train_loader):
#    print(batch_idx, waveforms, labels_batch)
#    break

In [ ]:
for batch_idx, (waveforms, labels_batch) in enumerate(train_loader):
   print(batch_idx, waveforms, labels_batch)
   break

In [5]:
import torch

In [6]:
class CNNModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.spec = T.MelSpectrogram(n_fft=1024, sample_rate=16000, n_mels=64, hop_length=512)

        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2) 
        )
        

        self.pool = torch.nn.AdaptiveAvgPool2d((4, 4)) 
        
        self.flatten = torch.nn.Flatten()
        
        self.dense = torch.nn.Linear(64 * 4 * 4, 35)

    def forward(self, x):
        x = self.spec(x)
        x = torch.log(x + 1e-10)

        #Permet de passer un seul élément pendant la phase de production
        if len(x.shape) < 3:
            x = x.unsqueeze(0)

        x = self.cnn(x)

        x = self.pool(x)
        x = self.flatten(x)
        return self.dense(x)

In [ ]:
import os
print(os.cpu_count())

In [12]:
import torch.optim as optim
import torchaudio.transforms as T

device = "cuda"

model = CNNModule()
model.to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.train() 

epochs = 3
best_loss = float('inf')
patience = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (waveforms, labels_batch) in enumerate(train_loader):
        waveforms = waveforms.to(device)
        labels_batch = labels_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(waveforms)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for waveforms, labels_batch in val_loader:
            waveforms = waveforms.to(device)
            labels_batch = labels_batch.to(device)
            outputs = model(waveforms)
            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = correct / total
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    if val_loss < best_loss:
        best_loss = val_loss
        trigger_times = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Nouveau meilleur modèle sauvegardé (Validation Loss: {best_loss:.4f})")
    else:
        trigger_times += 1
        print(f"Pas d'amélioration. Compteur Early Stopping: {trigger_times}/{patience}")
        if trigger_times >= patience:
            print(f"Early Stopping déclenché ! Arrêt à l'époque {epoch+1}")
            break

RuntimeError: DataLoader worker (pid(s) 367756, 384256) exited unexpectedly

In [ ]:
###Pour évaluer les performances

def eval_performance(model_to_eval, dataset_eval):
    model_to_eval.eval()
    correct = 0
    softmax = torch.nn.Softmax(dim=1)

    with torch.no_grad():
        for i in range(0,len(dataset_eval)):
            elem = dataset_eval.__getitem__(i)

            output = model_to_eval(elem[0])
            res = softmax(output)

            if res.argmax(dim=1).item() == labels.index(elem[2]): correct = correct + 1

    print("Accuracry : ", correct / len(dataset_eval) * 100, "%")
    return correct / len(dataset_eval)

#eval_performance(model, data_validation)


In [ ]:
%pip install sounddevice
%pip install scipy

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

"""
print(sd.query_devices())
device_info = sd.query_devices(10)
print(device_info)
"""

model = CNNModule()

model.load_state_dict(torch.load("./best_model.pth", weights_only=True))


device_index = 10

sd.default.device = device_index

duration = 1 
fs = int(sd.query_devices(device_index)["default_samplerate"])
filename = "capture.wav"

print("Enregistrement en cours...")
audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait() 
print("Enregistrement terminé.")

audio = torch.from_numpy(audio.T).float()

if fs != 16000 : 
    resampler = torchaudio.transforms.Resample(orig_freq=fs, new_freq=16000)
    audio = resampler(audio)


model.eval()

softmax_eval_live = torch.nn.Softmax(dim=1)

plt.figure(figsize=(20,5))
plt.xticks(rotation=90)
plt.title("Distribution")
plt.bar(labels,softmax_eval_live(model(audio))[0].detach().numpy())

write(filename, 16000, audio.T.numpy())
print(f"Audio sauvegardé dans {filename}")



In [ ]:
#Rajouter les dataloader en parametres
def train_model(
    device,
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    epochs=3,
    patience=5,
    save_path="best_model"
):
    """
    Cette méthode permet d'entraîner le modèle avec un earlystopping
    """
    model.to(device)

    best_loss = float('inf')
    trigger_times = 0

    for epoch in range(epochs):
        # Entraînement du modèle sur les données d'entraînement
        model.train()
        running_loss = 0.0

        for batch_idx, (waveforms, labels_batch) in enumerate(train_loader):

            waveforms = waveforms.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            outputs = model(waveforms)

            loss = criterion(outputs, labels_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 50 == 0:
                print(
                    f"Epoch {epoch+1}/{epochs} | "
                    f"Batch {batch_idx}/{len(train_loader)} | "
                    f"Loss: {loss.item():.4f}"
                )

        train_loss = running_loss / len(train_loader)

        # Evaluation du modèle sur les données de validation
        model.eval()

        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for waveforms, labels_batch in val_loader:

                waveforms = waveforms.to(device)
                labels_batch = labels_batch.to(device)

                outputs = model(waveforms)

                loss = criterion(outputs, labels_batch)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)

                total += labels_batch.size(0)
                correct += (predicted == labels_batch).sum().item()

        val_loss /= len(val_loader)
        val_acc = correct / total

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        # Ici on vérifie le early-stopping
        if val_loss < best_loss:

            best_loss = val_loss
            trigger_times = 0

            torch.save(model.state_dict(), save_path + model.__name__ + ".pth")

            print(
                f"Nouveau meilleur modèle sauvegardé "
                f"(Validation Loss: {best_loss:.4f})"
            )

        else:

            trigger_times += 1

            print(
                f"Pas d'amélioration. "
                f"Compteur Early Stopping: {trigger_times}/{patience}"
            )

            if trigger_times >= patience:
                print(f"Early Stopping déclenché à l'époque {epoch+1}")
                break
    #Rajouter l'affichage du temps d'entrainement total
    return model

In [ ]:
import torch
import torchaudio.transforms as T

class DoubleCNNModule(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.spec = T.MelSpectrogram(
            sample_rate=16000,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )

        self.cnn1D = torch.nn.Sequential(
            torch.nn.Conv1d(1, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(2),

            torch.nn.Conv1d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(2)
        )

        self.pool1D = torch.nn.AdaptiveAvgPool1d(16)

        self.cnn2D = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )

        self.pool2D = torch.nn.AdaptiveAvgPool2d((4, 4))

        self.flatten = torch.nn.Flatten()

        self.dense = torch.nn.Linear(2048, 35)

    def forward(self, x):

        #Permet de rajouter la dimension du batch pour un échantillon seul
        if len(x.shape) < 3:
            x = x.unsqueeze(0)

        x_1D = self.cnn1D(x)
        x_1D = self.pool1D(x_1D)
        x_1D = self.flatten(x_1D)

        x_2D = self.spec(x) 
        x_2D = torch.log(x_2D + 1e-10)

        x_2D = self.cnn2D(x_2D)
        x_2D = self.pool2D(x_2D)
        x_2D = self.flatten(x_2D)

        x = torch.cat([x_1D, x_2D], dim=1)

        return self.dense(x)




In [ ]:
from torch import optim

model = DoubleCNNModule()

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model("cpu", model, criterion, optimizer, train_loader, val_loader)

In [ ]:
class CNNBiLSTM(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.spec = T.MelSpectrogram(
            sample_rate=16000,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )

        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )

        self.lstm = torch.nn.LSTM(
            input_size=64 * 16, 
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = torch.nn.Linear(128 * 2, 35)

    def forward(self, x):
        x = self.spec(x)
        x = torch.log(x + 1e-10)

        if len(x.shape) <= 3:
            x = x.unsqueeze(0)

        x = self.cnn(x)

        """B, C, F, T = x.shape

        x = x.permute(0, 3, 1, 2)
        x = x.reshape(B, T, C * F)

        x, (h_n, c_n) = self.lstm(x)

        # Dernier état du BiLSTM
        forward = h_n[-2]
        backward = h_n[-1]

        x = torch.cat([forward, backward], dim=1)"""

        return self.fc(x)

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import torchaudio
import matplotlib.pyplot as plt


model = DoubleCNNModule()

model.load_state_dict(torch.load("./best_model_doubleCNN.pth", weights_only=True))


device_index = 10

sd.default.device = device_index

duration = 1 
fs = int(sd.query_devices(device_index)["default_samplerate"])
filename = "capture.wav"

print("Enregistrement en cours...")
audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait() 
print("Enregistrement terminé.")

audio = torch.from_numpy(audio.T).float()

if fs != 16000 : 
    resampler = torchaudio.transforms.Resample(orig_freq=fs, new_freq=16000)
    audio = resampler(audio)


model.eval()

softmax_eval_live = torch.nn.Softmax(dim=1)

plt.figure(figsize=(20,5))
plt.xticks(rotation=90)
plt.title("Distribution")
plt.bar(labels,softmax_eval_live(model(audio))[0].detach().numpy())

write(filename, 16000, audio.T.numpy())
print(f"Audio sauvegardé dans {filename}")



### Hyperparameter Tuning

In [ ]:
import torch
import torchaudio.transforms as T

class CNNModule(torch.nn.Module):
    def __init__(
        self,
        n_mels,
        conv1_channels,
        conv2_channels,
        kernel_size
    ):
        super().__init__()

        self.spec = T.MelSpectrogram(
            n_fft=1024,
            sample_rate=16000,
            n_mels=n_mels,
            hop_length=512
        )

        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(
                1,
                conv1_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2
            ),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(
                conv1_channels,
                conv2_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2
            ),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )

        self.pool = torch.nn.AdaptiveAvgPool2d((4, 4))

        self.flatten = torch.nn.Flatten()

        self.dense = torch.nn.Linear(
            conv2_channels * 4 * 4,
            35
        )

    def forward(self, x):
        x = self.spec(x)
        x = torch.log(x + 1e-10)

        if len(x.shape) < 3:
            x = x.unsqueeze(0)

        x = self.cnn(x)
        x = self.pool(x)
        x = self.flatten(x)

        return self.dense(x)

In [ ]:
import torch
import optuna

def objective(trial):

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)

    conv1_channels = trial.suggest_categorical("conv1_channels", [16, 32, 64])
    conv2_channels = trial.suggest_categorical("conv2_channels", [32, 64, 128])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5])
    n_mels = trial.suggest_categorical("n_mels", [40, 64, 80])

    train_loader = DataLoader(
        data_train, 
        batch_size=32, 
        shuffle=True,
        collate_fn=pre_process_batch 
    )

    val_loader = DataLoader(
        data_validation, 
        batch_size=32, 
        shuffle=True,
        collate_fn=pre_process_batch 
    )

    model = CNNModule(
        conv1_channels=conv1_channels,
        conv2_channels=conv2_channels,
        kernel_size=kernel_size,
        n_mels=n_mels
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0
    patience = 2
    trigger = 0
    epochs = 3 

    for epoch in range(epochs):


        model.train()
        for waveforms, labels in train_loader:
            waveforms, labels = waveforms.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(waveforms)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for waveforms, labels in val_loader:
                waveforms, labels = waveforms.to(device), labels.to(device)

                outputs = model(waveforms)
                preds = outputs.argmax(1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total

        trial.report(val_acc, epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            trigger = 0
        else:
            trigger += 1
            if trigger >= patience:
                break

    return best_val_acc

In [ ]:
import optuna

device = "cpu"

study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=50
)